# Consulta a um DataFrame

Vamos entender como consultar DataFrames. A primeira etapa do processo é entender a máscara booleana. <br>
Um mascaramento booleano é a essência da primeira consulta rápida e eficiente tanto no NumPy quanto no Pandas.

LEMBRANDO: Uma máscara booleana é um array/matriz em que cada um dos valores do array é verdadeiro ou falso. Essa matriz é essencialmente sobreposta à outra estrutura de dados que estamos consultando, e qualquer célula alinhada com o valor verdadeiro será passada como resultado.

In [3]:
# Importando a biblioteca
import pandas as pd

In [4]:
# Vamos carregar o arquivo CSV
df = pd.read_csv('datasets/Admission_Predict.csv', index_col = 0)
df.head()

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
Serial No.,,,,,,,,
1,337,118,4,4.5,4.5,9.65,1,0.92
2,324,107,4,4.0,4.5,8.87,1,0.76
3,316,104,3,3.0,3.5,8.00,1,0.72
4,322,110,3,3.5,2.5,8.67,1,0.80
5,314,103,2,2.0,3.0,8.21,0,0.65


In [5]:
# Vamos limpar e corrigir colunas mal nomeadas com compreensão de listas
colunas = list(df.columns)
colunas = [x.strip() for x in colunas]

colunas[3] = 'Statement of Pupose'
colunas[4] = 'Letter of Recommendation'

df.columns = colunas
df.head()

,GRE Score,TOEFL Score,University Rating,Statement of Pupose,Letter of Recommendation,CGPA,Research,Chance of Admit
Serial No.,,,,,,,,
1,337,118,4,4.5,4.5,9.65,1,0.92
2,324,107,4,4.0,4.5,8.87,1,0.76
3,316,104,3,3.0,3.5,8.00,1,0.72
4,322,110,3,3.5,2.5,8.67,1,0.80
5,314,103,2,2.0,3.0,8.21,0,0.65


In [6]:
# As máscaras booleanas são criadas aplicando os operadores diretamente aos objetos do Pandas, isto é, Séries ou DataFrames.

# Por exemplo, estamos interessados em ver apenas os alunos que têm chances de admissão maior que 0.7.
mascara_booleana = df[['Chance of Admit']] > 0.7    # Coloquei em [] a mais para me passar no formato de DataFrame e não de Série, uma vez estando em uma dimensão será retornado série.
mascara_booleana

,Chance of Admit
Serial No.,
1,True
2,True
3,True
4,True
5,False
...,...
396,True
397,True
398,True


Perceba que o resultado do broadcasting de um operador de comparação é uma máscara booleana, com valores verdadeiros ou falsos, dependendo dos resultados da comparação. <br>
Por baixo, o Pandas está aplicando o operador de comparação que foi especificado por meio de vetorização, portanto, ocorre em paralelo e de forma eficiente.

In [7]:
# Após a criação da máscara booleana, podemos colocá-la sobre os dados e ocultar os dados que não queremos, os quais são representados por todos os valores falsos.
# Faremos isso usando a função .where() no DataFrame.

df.where(mascara_booleana)

,GRE Score,TOEFL Score,University Rating,Statement of Pupose,Letter of Recommendation,CGPA,Research,Chance of Admit
Serial No.,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.92
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.76
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.72
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.80
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
396,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.82
397,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.84
398,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.91


Portanto, vemos que o DataFrame resultante mantém os valores do índice original e apenas os dados que atendem à condição foram mostrados, de modo que todas as linhas que não atenderam à condição possuem dados NaN, mas essas linhas NÃO foram descartadas do nosso conjunto de dados.

In [8]:
# Mas e se não quisermosdados NaN? 
# Podemos usar a função dropna()
df.where(mascara_booleana).dropna().head()

,GRE Score,TOEFL Score,University Rating,Statement of Pupose,Letter of Recommendation,CGPA,Research,Chance of Admit
Serial No.,,,,,,,,


Assim, os dados que não satisfizeram a condição e ficaram com NaN são descartados, mas perceba que primeiro foi filtrado, os dados que não satisfizeram ficaram com NaN e só depois esses dados foram removidos. Isso perde eficiencia.

Apesar de bastante útil, a função .where() não é muito usada, pois os desenvolvedores do Pandas criaram uma sintaxe abreviada que combina where() e dropna(), representando ambos ao mesmo tempo sem perder a eficiência.

In [9]:
# Dá para filtrar de forma direta sem perder a eficiência.
# Basta utilizar a máscara booleana dentro do operador de indexação diretamente, como se fosse aplicar em uma matriz. 
df[mascara_booleana].head()

,GRE Score,TOEFL Score,University Rating,Statement of Pupose,Letter of Recommendation,CGPA,Research,Chance of Admit
Serial No.,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.92
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.76
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.72
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.80
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Vamos entender a combinação de várias máscaras booleanas, tais como múltiplos critérios para inclusão. 
# Isso é feito com "and", caso ambas as máscaras devam ser verdadeiras para queum valor verdadeiro esteja na máscara final, ou "or", se apenas uma máscara precisa ser verdadeira 

# Isso não é muito natural em Pandas, por exemplo, se quisermos pegar duas Séries booleanas e usar o "and" junto, fazemos:

mascara_booleana = (df[['Chance of Admit']] > 0.7) and (df[['Chance of Admit']] < 0.9)

# Isso vai dar erro

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [15]:
# Esse problema é porque você tem um objeto de Séries, e o Python, por trás, não sabe como comparar duas Séries usando "and" ou "or". Em vez disso, o autor do Pandas os substituiu por "|" e "&" para lidarmos com isso.

# JEITO CERTO
mascara_booleana = (df[['Chance of Admit']] > 0.7) & (df[['Chance of Admit']] < 0.9)
mascara_booleana


,Chance of Admit
Serial No.,
1,False
2,True
3,True
4,True
5,False
...,...
396,True
397,True
398,False


##### ATENÇÃO: Um erro comum é fazer comparações booleanas usando o operador "&" mas não colocar o "()" em torno dos termos 

In [16]:
# Outra forma de fazer isso é simplesmente se livrar completamente do operador de comparação e, em vez disso, usar as funções built in que imitam essa abordagem
mascara_booleana = df[['Chance of Admit']].gt(0.7) & df[['Chance of Admit']].lt(0.9)
mascara_booleana

,Chance of Admit
Serial No.,
1,False
2,True
3,True
4,True
5,False
...,...
396,True
397,True
398,False


- .gt() → greater than → maior que (>)
- .lt() → less than → menor que (<)

In [ ]:
# Podemos fazer de forma encadeada
mascara_booleana = df['Chance of Admit'].gt(0.7).lt(0.9)
mascara_booleana

Serial No.
1      False
2      False
3      False
4      False
5       True
       ...  
396    False
397    False
398    False
399     True
400    False
Name: Chance of Admit, Length: 400, dtype: bool